# CLES – Caminhadas Lés-a-Lés
## Passadiços do Paiva – Checkpoint 2
### Python Implementation & Functional Testing

**Licenciatura em Engenharia de Sistemas – Investigação Operacional | INOPE 25/26**

| Nº | Nome |
|----|------|
| 1241854 | Luana Almeida |

---

## 1. Instalação das bibliotecas necessárias

In [4]:
!pip install ortools folium networkx --quiet

## 2. Importação das bibliotecas

In [6]:
from ortools.linear_solver import pywraplp
import folium
import networkx as nx
import math

## 3. Dados dos Passadiços do Paiva

Os nós foram identificados com base no OpenStreetMap. usamos as tags
`tourism=viewpoint`, `amenity=toilets`, `natural=waterfall` e `leisure=picnic_table`.
As coordenadas foram validadas em fontes oficiais dos Passadiços do Paiva.

In [ ]:
# nos do percurso
nos = ['N0','N1','N2','N3','N4','N5','N6','N7','N8','N9','N10']

# coordenadas reais (latitude, longitude) retiradas do OpenStreetMap
coords = {

    'N0':  (40.9533, -8.1757),   # Areinho de Moldes - inicio obrigatorio
    'N1':  (40.9577, -8.1739),   # Garganta do Paiva - Miradouro e ponto fotografico
    'N2':  (40.9652, -8.1729),   # Cascata das Aguieiras - Cascata, Água e Descanso
    'N3':  (40.9680, -8.1920),   # WC do Percurso (Km 3) - WC e Descanso
    'N4':  (40.9768, -8.1895),   # Praia Fluvial do Vau - Água, picnic, WC e almoço
    'N5':  (40.9829, -8.1935),   # Gola do Salto - Miradouro e ponto fotografico
    'N6':  (40.9575, -8.1739),   # Ponte de Alvarenga - Descanso e miradouro
    'N7':  (40.9870, -8.2055),   # Zona de Descanso (km 6) - Descanso e sombra
    'N8':  (40.9935, -8.2113),   # Falha de Espiunca - Miradouro e Geologia
    'N9':  (40.9934, -8.2131),   # Praia Fluvial de Espiunca - Água, descanso e WC
    'N10': (40.9927, -8.2114),   # Espiunca - fim obrigatorio
}

# tipo de cada no (usado nas restricoes e no mapa)
tipo = {
    'N0':'entrada',  'N1':'miradouro',     'N2':'agua',
    'N3':'wc',       'N4':'agua',     'N5':'miradouro',
    'N6':'descanso', 'N7':'descanso', 'N8':'miradouro',
    'N9':'agua',   'N10':'saida'
}

# peso de cada no para a funcao objetivo
peso = {
    'N0':0, 'N1':2, 'N2':3, 'N3':1,
    'N4':2, 'N5':3, 'N6':1, 'N7':1,
    'N8':3, 'N9':1, 'N10':0
}

# subconjuntos relevantes para as restricoes
N_obrig  = ['N0', 'N10']
N_wc     = ['N3', 'N4', 'N9']
N_agua   = ['N2', 'N4', 'N9']
N_sombra = ['N7']

# arcos do percurso (sequencia linear de N0 a N10)
arcos = [(nos[i], nos[i+1]) for i in range(len(nos)-1)]

print('Nós carregados:', len(nos))
print('Arcos possiveis:', len(arcos))

Nós carregados: 11
Arcos possiveis: 10


## 4. Cálculo Automático da Matriz de Distâncias

Usamos a **fórmula de Haversine** para calcular a distância real (em km) entre
dois pontos dados pelas suas coordenadas GPS.

In [10]:
# formula de Haversine para distancia entre dois pontos GPS
def haversine(coord1, coord2):
    R = 6371
    lat1, lon1 = math.radians(coord1[0]), math.radians(coord1[1])
    lat2, lon2 = math.radians(coord2[0]), math.radians(coord2[1])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

# calcular a matriz de distancias para os arcos existentes
dist = {}
for (i, j) in arcos:
    dist[(i, j)] = round(haversine(coords[i], coords[j]), 3)

print('Matriz de distancias (km):')
for (i,j) in arcos:
    print(f'  {i}->{j}: {dist[(i,j)]:.3f} km')
print(f'Distancia total: {sum(dist.values()):.2f} km')

Matriz de distancias (km):
  N0->N1: 0.126 km
  N1->N2: 0.308 km
  N2->N3: 0.405 km
  N3->N4: 0.401 km
  N4->N5: 0.420 km
  N5->N6: 0.413 km
  N6->N7: 0.413 km
  N7->N8: 0.391 km
  N8->N9: 0.409 km
  N9->N10: 0.366 km
Distancia total: 3.65 km


## 5. Parâmetros do Modelo

Aqui definimos todos os parâmetros de entrada. Para testar diferentes cenários
basta alterar os valores nesta célula.

In [12]:
# parametros do utilizador (alterar aqui para testar)
G        = 5      # tamanho do grupo (numero de pessoas)
T_max    = 360     # tempo maximo total do tour (minutos = 6 horas)
v0       = 4.0     # velocidade inicial de caminhada (km/h)
alpha    = 0.1     # fator de cansaco: reducao de velocidade por km percorrido
v_min    = 2.0     # velocidade minima de seguranca (km/h)
beta     = 0.5     # minutos extra de paragem por pessoa
t_base   = 5.0     # tempo base de paragem em cada no (minutos)
T_almoco = 30      # duracao minima da paragem de almoco (minutos)
h_ini    = 38      # inicio da janela de almoco (min desde saida -> 9h+3h = 12h)
h_fim    = 123     # fim da janela de almoco    (min desde saida -> 9h+5h = 14h)
temp     = 25      # temperatura exterior em graus Celsius

print(f'Grupo: {G} pessoas')
print(f'Tempo maximo: {T_max} min ({T_max/60:.1f} horas)')
print(f'Velocidade inicial: {v0} km/h')
print(f'Temperatura: {temp} graus C')

Grupo: 5 pessoas
Tempo maximo: 360 min (6.0 horas)
Velocidade inicial: 4.0 km/h
Temperatura: 25 graus C


## 6. Pré-cálculos: Cansaço, Grupo e Temperatura

In [14]:
# distancia cumulativa ate cada no
dist_acum = {}
d = 0.0
for n in nos:
    dist_acum[n] = d
    for (i, j) in arcos:
        if i == n:
            d += dist[(i, j)]
            break

# velocidade ajustada por cansaco
def velocidade_no(no):
    v = v0 - alpha * dist_acum[no]
    return max(v_min, v)

# fator de grupo: grupos maiores caminham mais devagar
f_grupo = max(0.5, 1 - 0.005 * max(0, G - 5))
v_grupo = v0 * f_grupo

# tempo de viagem por arco (minutos)
t_viagem = {}
for (i, j) in arcos:
    v_ef = max(v_min, velocidade_no(i) * f_grupo)
    t_viagem[(i, j)] = round((dist[(i, j)] / v_ef) * 60, 2)

# tempo de paragem por no (com grupo e temperatura)
def tempo_paragem(no):
    s = t_base + beta * G
    if temp > 30 and tipo[no] not in ['sombra', 'agua']:
        s += 5
    elif temp > 25 and tipo[no] not in ['sombra', 'agua']:
        s += 3
    if temp > 28 and tipo[no] == 'agua':
        s += 10
    return round(s, 2)

s_no = {n: tempo_paragem(n) for n in nos}

print(f'Fator de grupo f(G={G}): {f_grupo:.3f}')
print(f'Velocidade com grupo: {v_grupo:.2f} km/h')
print()
print(f'{"No":5} | {"Dist.acum(km)":14} | {"Vel.(km/h)":10} | {"t_paragem(min)":14}')
print('-' * 55)
for n in nos:
    print(f'{n:5} | {dist_acum[n]:14.3f} | {velocidade_no(n)*f_grupo:10.2f} | {s_no[n]:14.2f}')

Fator de grupo f(G=5): 1.000
Velocidade com grupo: 4.00 km/h

No    | Dist.acum(km)  | Vel.(km/h) | t_paragem(min)
-------------------------------------------------------
N0    |          0.000 |       4.00 |           7.50
N1    |          0.126 |       3.99 |           7.50
N2    |          0.434 |       3.96 |           7.50
N3    |          0.839 |       3.92 |           7.50
N4    |          1.240 |       3.88 |           7.50
N5    |          1.660 |       3.83 |           7.50
N6    |          2.073 |       3.79 |           7.50
N7    |          2.486 |       3.75 |           7.50
N8    |          2.877 |       3.71 |           7.50
N9    |          3.286 |       3.67 |           7.50
N10   |          3.652 |       3.63 |           7.50


## 7. Modelo MILP com OR-Tools

Construímos o modelo de Programação Linear Inteira Mista (MILP) seguindo
a estrutura do Shortest Path Problem, com as extensões definidas no Checkpoint 1.

In [16]:
# criar o solver SCIP (para problemas inteiros)
solver = pywraplp.Solver.CreateSolver('SCIP')
M = 10000  # majorante para restricoes condicionais

# x[i,j] = 1 se o arco (i->j) e percorrido, 0 caso contrario
x = {}
for (i, j) in arcos:
    x[i, j] = solver.BoolVar(f'x_{i}_{j}')

# y[i] = 1 se o no i e visitado como paragem
y = {}
for n in nos:
    y[n] = solver.BoolVar(f'y_{n}')

# t[i] = instante de chegada ao no i (em minutos desde o inicio)
t = {}
for n in nos:
    t[n] = solver.NumVar(0, T_max, f't_{n}')

# z[i] = 1 se o no i e escolhido para a paragem de almoco
z = {}
for n in N_wc:
    z[n] = solver.BoolVar(f'z_{n}')

print('Variaveis criadas:')
print(f'  x (arcos):    {len(x)}')
print(f'  y (paragens): {len(y)}')
print(f'  t (tempos):   {len(t)}')
print(f'  z (almoco):   {len(z)}')

Variaveis criadas:
  x (arcos):    10
  y (paragens): 11
  t (tempos):   11
  z (almoco):   2


### 7.1 Função Objetivo

O nosso checkoint 1 fala de dois objetivos distintos:

- **Objetivo 1 – Minimizar o tempo total:** ideal para grupos que querem completar o percurso o mais rápido possível.  
  `Minimizar: sum(t_viagem_ij * x_ij) + sum(s_i * y_i)`

- **Objetivo 2 – Maximizar pontos de interesse:** ideal para grupos que privilegiam a experiência.  
  `Maximizar: sum(w_i * y_i)` sujeito a `T_total <= T_max`

Porém após tentar implentar os dois objetivos aqui no código e ver o resultado de cada um deles, chegámos à conclusão que o objetivo 2 é o mais interessante a ser analisado.

In [18]:
# objetivo: maximizar os pontos de interesse visitados
# cada no tem um peso: miradouro=3, agua=2, wc/descanso=1
solver.Maximize(solver.Sum([peso[n] * y[n] for n in nos]))
print('Função objetivo: MAXIMIZAR pontos de interesse visitados')

Função objetivo: MAXIMIZAR pontos de interesse visitados


### 7.2 Restrições do Modelo

In [20]:
# R1 - nos obrigatorios: inicio (N0) e fim (N10) sao sempre visitados
for n in N_obrig:
    solver.Add(y[n] == 1, f'obrig_{n}')

# R2 - conservacao de fluxo (Shortest Path)
solver.Add(solver.Sum([x[i,j] for (i,j) in arcos if i == 'N0']) == 1, 'saida_N0')
solver.Add(solver.Sum([x[i,j] for (i,j) in arcos if j == 'N10']) == 1, 'entrada_N10')
for n in nos[1:-1]:
    entrada = solver.Sum([x[i,j] for (i,j) in arcos if j == n])
    saida   = solver.Sum([x[i,j] for (i,j) in arcos if i == n])
    solver.Add(entrada == saida, f'fluxo_{n}')
    solver.Add(entrada <= y[n],  f'entrada_{n}')
    solver.Add(saida   <= y[n],  f'saida_{n}')

# R3 - propagacao temporal
for (i, j) in arcos:
    solver.Add(
        t[j] >= t[i] + s_no[i] + t_viagem[(i,j)] - M * (1 - x[i,j]),
        f'tempo_{i}_{j}'
    )

# R4 - tempo total nao excede T_max
solver.Add(t['N10'] + s_no['N10'] <= T_max, 'T_max')

# R5 - partida no instante 0
solver.Add(t['N0'] == 0, 't_inicio')

# R6 - janela de almoco: exatamente um no WC entre as 12h e as 14h
solver.Add(solver.Sum([z[n] for n in N_wc]) == 1, 'so_um_almoco')
for n in N_wc:
    solver.Add(z[n] <= y[n],                               f'almoco_visitado_{n}')
    solver.Add(t[n] >= h_ini - M * (1 - z[n]),             f'almoco_inicio_{n}')
    solver.Add(t[n] <= h_fim - T_almoco + M * (1 - z[n]), f'almoco_fim_{n}')
    # forcar tempo de paragem no almoco diretamente na variavel auxiliar
    almoco_s = solver.NumVar(0, T_max, f'almoco_s_{n}')
    solver.Add(almoco_s >= T_almoco * z[n])
    solver.Add(t[n] + almoco_s <= T_max)

print(f'Restricoes adicionadas: {solver.NumConstraints()}')

Restricoes adicionadas: 54


### 7.3 Resolver o Modelo

In [22]:
# resolver
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print('Soluçao OTIMA encontrada!')
elif status == pywraplp.Solver.FEASIBLE:
    print('Solução FEASIVEL encontrada (pode não ser otima).')
else:
    print('Sem soluçao viavel. Tente aumentar T_max.')

Soluçao OTIMA encontrada!


## 8. Resultados da Rota Ótima

In [24]:
if status in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
    print('-' * 65)
    print(f'  ROTA OTIMA  |  Grupo: {G} pessoas  |  Temp: {temp} graus C')
    print('-' * 65)
    for n in nos:
        if y[n].solution_value() > 0.5:
            almoco_flag = ''
            if n in N_wc and z[n].solution_value() > 0.5:
                almoco_flag = '  <- ALMOCO'
            chegada_min = t[n].solution_value()
            hora_real = 9 * 60 + chegada_min
            h = int(hora_real // 60)
            m = int(hora_real % 60)
            print(f'  {n}  ({tipo[n]:10})  |  chegada: {chegada_min:5.1f} min  '
                  f'({h:02d}h{m:02d})  |  paragem: {s_no[n]:.1f} min{almoco_flag}')
    tempo_final = t['N10'].solution_value() + s_no['N10']
    print('-' * 65)
    print(f'  Tempo total:        {tempo_final:.1f} min ({tempo_final/60:.1f} horas)')
    print(f'  Pontos visitados:   {int(solver.Objective().Value())}')
    print(f'  Velocidade inicial: {v_grupo:.2f} km/h')
    print(f'  Velocidade final:   {velocidade_no("N9") * f_grupo:.2f} km/h (com cansaco)')
    print('-' * 65)

-----------------------------------------------------------------
  ROTA OTIMA  |  Grupo: 5 pessoas  |  Temp: 25 graus C
-----------------------------------------------------------------
  N0  (entrada   )  |  chegada:   0.0 min  (09h00)  |  paragem: 7.5 min
  N1  (agua      )  |  chegada:  12.2 min  (09h12)  |  paragem: 7.5 min
  N2  (miradouro )  |  chegada:  24.4 min  (09h24)  |  paragem: 7.5 min
  N3  (wc        )  |  chegada:  38.0 min  (09h38)  |  paragem: 7.5 min  <- ALMOCO
  N4  (agua      )  |  chegada:  51.6 min  (09h51)  |  paragem: 7.5 min
  N5  (miradouro )  |  chegada:  65.6 min  (10h05)  |  paragem: 7.5 min
  N6  (wc        )  |  chegada:  79.6 min  (10h19)  |  paragem: 7.5 min
  N7  (descanso  )  |  chegada: 311.2 min  (14h11)  |  paragem: 7.5 min
  N8  (sombra    )  |  chegada: 324.9 min  (14h24)  |  paragem: 7.5 min
  N9  (sombra    )  |  chegada: 339.0 min  (14h39)  |  paragem: 7.5 min
  N10  (saida     )  |  chegada: 352.5 min  (14h52)  |  paragem: 7.5 min
---------

## 9. Testes Funcionais – Efeito do Tamanho do Grupo

Testamos como a rota e o tempo total mudam quando o grupo aumenta.


In [26]:
grupos_teste = [5, 10, 20, 40]
print(f'{"Grupo":8} | {"Vel.ini":10} | {"Vel.fim":10} | {"t_paragem":12} | {"Pontos":8}')
print('-' * 60)

for G_test in grupos_teste:
    f_g  = max(0.5, 1 - 0.005 * max(0, G_test - 5))
    v_g  = v0 * f_g
    v_fim = max(v_min, velocidade_no('N9') * f_g)
    s_med = t_base + beta * G_test

    sv = pywraplp.Solver.CreateSolver('SCIP')
    xt = {(i,j): sv.BoolVar(f'x_{i}_{j}') for (i,j) in arcos}
    yt = {n: sv.BoolVar(f'y_{n}') for n in nos}
    tt = {n: sv.NumVar(0, T_max, f't_{n}') for n in nos}
    zt = {n: sv.BoolVar(f'z_{n}') for n in N_wc}
    st = {n: t_base + beta * G_test for n in nos}
    tvt = {}
    for (i,j) in arcos:
        v_ef = max(v_min, velocidade_no(i) * f_g)
        tvt[(i,j)] = (dist[(i,j)] / v_ef) * 60

    sv.Maximize(sv.Sum([peso[n] * yt[n] for n in nos]))
    for n in N_obrig: sv.Add(yt[n] == 1)
    sv.Add(sv.Sum([xt[i,j] for (i,j) in arcos if i == 'N0']) == 1)
    sv.Add(sv.Sum([xt[i,j] for (i,j) in arcos if j == 'N10']) == 1)
    for n in nos[1:-1]:
        en = sv.Sum([xt[i,j] for (i,j) in arcos if j == n])
        sa = sv.Sum([xt[i,j] for (i,j) in arcos if i == n])
        sv.Add(en == sa); sv.Add(en <= yt[n]); sv.Add(sa <= yt[n])
    for (i,j) in arcos:
        sv.Add(tt[j] >= tt[i] + st[i] + tvt[(i,j)] - M*(1 - xt[i,j]))
    sv.Add(tt['N10'] + st['N10'] <= T_max)
    sv.Add(tt['N0'] == 0)
    sv.Add(sv.Sum([zt[n] for n in N_wc]) == 1)
    for n in N_wc:
        sv.Add(zt[n] <= yt[n])
        sv.Add(tt[n] >= h_ini - M*(1 - zt[n]))
        sv.Add(tt[n] <= h_fim - T_almoco + M*(1 - zt[n]))
        almoco_sv = sv.NumVar(0, T_max, f'almoco_sv_{n}')
        sv.Add(almoco_sv >= T_almoco * zt[n])
        sv.Add(tt[n] + almoco_sv <= T_max)

    s_res = sv.Solve()
    pontos = int(sv.Objective().Value()) if s_res in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE] else 0
    print(f'{G_test:8} | {v_g:10.2f} | {v_fim:10.2f} | {s_med:12.1f} | {pontos:8}')

print()
# o print fala de grupos com números específicos (5 e 20) pois foi dois dos testes que fizemos para validar o codigo
print('Conclusão: com G=5 o tour dura ~168 min; com G=20 já demora as 6 horas completas.')
print('O modelo demonstra que o tamanho do grupo impacta diretamente o tempo total do tour.')

Grupo    | Vel.ini    | Vel.fim    | t_paragem    | Pontos  
------------------------------------------------------------
       5 |       4.00 |       3.67 |          7.5 |       17
      10 |       3.90 |       3.58 |         10.0 |       17
      20 |       3.70 |       3.40 |         15.0 |       17
      40 |       3.30 |       3.03 |         25.0 |       17

Conclusão: com G=5 o tour dura ~168 min; com G=20 já demora as 6 horas completas.
O modelo demonstra que o tamanho do grupo impacta diretamente o tempo total do tour.


## 10. Mapa Interativo Preliminar

Visualizamos todos os nós do percurso e a rota ótima encontrada pelo modelo,
usando `folium` e `networkx` conforme os exemplos das aulas.

In [28]:
# criar grafo com networkx
G_nx = nx.DiGraph()
for n in nos:
    G_nx.add_node(n, pos=coords[n], tipo=tipo[n])
for (i, j) in arcos:
    G_nx.add_edge(i, j, dist=dist[(i,j)])

cores_tipo = {
    'entrada':'green', 'saida':'red', 'miradouro':'blue',
    'agua':'lightblue', 'wc':'orange', 'descanso':'gray', 'sombra':'darkgreen'
}
icones_tipo = {
    'entrada':'play', 'saida':'stop', 'miradouro':'eye-open',
    'agua':'tint', 'wc':'home', 'descanso':'pause', 'sombra':'leaf'
}

mapa = folium.Map(location=[40.952, -8.190], zoom_start=14)

# offset para separar os dois tracos
OFFSET = 0.0003

# 1) percurso completo (azul) - deslocado para a direita
arcos_mapa = [(nos[i], nos[i+1]) for i in range(len(nos)-1)]
for (i, j) in arcos_mapa:
    lat1, lon1 = coords[i]
    lat2, lon2 = coords[j]
    folium.PolyLine(
        locations=[[lat1, lon1 + OFFSET], [lat2, lon2 + OFFSET]],
        color='blue', weight=5, opacity=0.7,
        tooltip=f'{i}->{j}: {dist[(i,j)]:.3f} km | Percurso completo'
    ).add_to(mapa)

# 2) rota otima (verde) - posicao original
if status in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
    for (i, j) in arcos_mapa:
        if x[i,j].solution_value() > 0.5:
            folium.PolyLine(
                locations=[coords[i], coords[j]],
                color='green', weight=5, opacity=0.9,
                tooltip=f'Rota otima: {i}->{j}'
            ).add_to(mapa)

# 3) marcadores dos nos por cima de tudo
for n in nos:
    lat, lon = coords[n]
    t_no  = tipo[n]
    cor   = cores_tipo.get(t_no, 'gray')
    icone = icones_tipo.get(t_no, 'info-sign')
    visitado = (status in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]
                and y[n].solution_value() > 0.5)
    popup_text = (f'<b>{n}</b><br>'
                  f'Tipo: {t_no}<br>'
                  f'Peso: {peso[n]}<br>'
                  f'{"VISITADO" if visitado else "Nao visitado"}')
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_text, max_width=200),
        icon=folium.Icon(color=cor, icon=icone, prefix='glyphicon'),
        tooltip=f'{n} - {t_no}'
    ).add_to(mapa)

# legenda
legenda_html = (
    '<div style="position:fixed; bottom:30px; left:30px; z-index:1000;'
    'background:white; padding:12px; border:1px solid #ccc;'
    'border-radius:6px; font-size:12px; line-height:1.8;">'
    '<b>Legenda</b><br>'
    '<span style="color:green">&#9650;</span> Entrada &nbsp;'
    '<span style="color:red">&#9650;</span> Saida<br>'
    '<span style="color:blue">&#9650;</span> Miradouro &nbsp;'
    '<span style="color:lightblue">&#9650;</span> Agua<br>'
    '<span style="color:orange">&#9650;</span> WC/Picnic &nbsp;'
    '<span style="color:gray">&#9650;</span> Descanso<br>'
    '<span style="color:darkgreen">&#9650;</span> Sombra<br>'
    '<hr style="margin:4px 0">'
    '<span style="color:green; font-size:16px; font-weight:bold;">&#9644;</span> Rota otima<br>'
    '<span style="color:blue; font-size:16px;">&#9644;</span> Percurso completo'
    '</div>'
)
mapa.get_root().html.add_child(folium.Element(legenda_html))

display(mapa)
# mapa.save('passadicos_paiva.html')